# DuckDB — Python Walkthrough

**YZV 322E — Tool Presentation, Spring 2026.**

This notebook is an alternative to the recorded video demo, which uses
the official DuckDB browser UI (`duckdb -ui`). The cells here cover the
same ideas — querying Parquet directly, mixing Pandas with SQL, using
DuckDB's analytical SQL extensions, but from inside Python.

**Before running:** make sure `bash scripts/00_download_data.sh` has
been executed so the three Parquet files are present in `../data/`.

In [ ]:
import os
import time

import duckdb
import pandas as pd

print('DuckDB version:', duckdb.__version__)
print('Working directory:', os.getcwd())

## 1. Query a Parquet file with one line of SQL

DuckDB reads the Parquet file glob in place. No load step, no schema
definition. The row count below scans only the file metadata.

In [ ]:
duckdb.sql("""
    SELECT COUNT(*) AS total_rows
    FROM '../data/yellow_2024-*.parquet'
""").df()

## 2. A real analytical query — top pickup zones by revenue

Group 9.5 million rows by pickup zone, compute trip count, average
fare, and total revenue. The whole thing runs in well under a second
on a laptop, in a single SQL statement, with no Python loop.

In [ ]:
t0 = time.perf_counter()

df = duckdb.sql("""
    SELECT
        PULocationID,
        COUNT(*)                              AS trips,
        ROUND(AVG(fare_amount)::NUMERIC, 2)   AS avg_fare,
        ROUND(SUM(total_amount)::NUMERIC, 2)  AS total_revenue
    FROM '../data/yellow_2024-*.parquet'
    GROUP BY PULocationID
    ORDER BY total_revenue DESC
    LIMIT 10
""").df()

print(f'Query time: {time.perf_counter() - t0:.3f} s')
df

## 3. SQL on a Pandas DataFrame

DuckDB can query a Pandas DataFrame directly by name, useful when you
already have a DataFrame in memory and want to write SQL against it
instead of chaining Pandas operations. Internally DuckDB reads the
DataFrame's columnar buffers via Apache Arrow, so the handoff is fast.

In [ ]:
small = pd.read_parquet('../data/yellow_2024-01.parquet').head(1000)

duckdb.sql("""
    SELECT
        payment_type,
        COUNT(*)         AS trips,
        AVG(tip_amount)  AS avg_tip
    FROM small
    GROUP BY payment_type
    ORDER BY payment_type
""").df()

## 4. PIVOT — DuckDB's analytical SQL extensions

DuckDB extends SQL with constructs that are awkward in standard SQL,
such as `PIVOT`. The query below shows trips per hour split into two
columns by payment type (1 = credit card, 2 = cash) — the kind of
shape that would normally need a Pandas `pivot_table` call.

In [ ]:
duckdb.sql("""
    PIVOT (
        SELECT
            EXTRACT(hour FROM tpep_pickup_datetime) AS hr,
            payment_type,
            COUNT(*) AS trips
        FROM '../data/yellow_2024-01.parquet'
        WHERE payment_type IN (1, 2)
        GROUP BY hr, payment_type
    )
    ON payment_type
    USING SUM(trips)
    ORDER BY hr
""").df()

---

For the recorded UI version of the demo (with the live Postgres
integration), see [`DEMO_SQL_CELLS.md`](../DEMO_SQL_CELLS.md) and the
video link in [`../video_link.txt`](../video_link.txt).